# demo

> Isolated rendering of markup with current styles and behaviors, for notebooks and docs

In [3]:
#| default_exp demo

The registry holds styles and behaviors, and the behavior runtime turns them into JavaScript. To develop a component, we need to see the result and try its interactions. `demo` renders ordinary FastHTML markup in an isolated document with the current assets. The same preview is used both during notebook authoring and when viewing generated docs.


In [5]:
from fastcore.utils import *
from fasthtml.common import to_xml, Div, Iframe, Button, Html, Head, Body, Meta, Style, Script
from fasthtml.core import htmxsrc
from answerui.registry import build_css
from answerui.behaviors import build_js

In [7]:
from answerui.registry import Rules, hypelt
from fasthtml.common import P, Details, Summary

## Rendering previews

A preview needs its own stylesheet and behavior script, as well as htmx. FastHTML exports its default script as `htmxsrc`, so we use that.

Both scripts load in the head, so the behavior listeners are installed before htmx fires `htmx:load` for the initial page. Supplying `css` replaces the registered stylesheet, which lets us experiment with styling independently of the rest of the library.

In [9]:
def demo_html(
    *fts,     # Markup to render
    css=None, # Stylesheet for the document; None uses the registry's build_css()
):
    "A standalone document containing markup, CSS, behaviors, and FastHTML's default htmx"
    head = Head(Meta(charset='utf-8'), Meta(name='viewport', content='width=device-width, initial-scale=1'),
        Style(build_css() if css is None else css), Script(build_js()), htmxsrc)
    return to_xml(Html(head, Body(*fts)))

For rendering in notebooks, we use an iframe to isolate the preview, so the styling of the notebook (e.g. solveit's CSS) can't affect the rendered markup, and vice versa. Its `srcdoc` attribute holds the complete document from `demo_html`. By default, the iframe automatically resizes to fit its contents, and we include a replay button so that we can reload dynamic components (e.g. ones that auto-dismiss).

In [11]:
_sizer = Script(r"new ResizeObserver(() => frameElement.style.height = document.body.scrollHeight+24+'px').observe(document.body)")

def demo(
    *fts,        # Markup to render
    height=None, # Fixed iframe height, e.g. '10rem', for top-layer content the frame can't measure; None follows the content
    css=None,    # Stylesheet for the document; None uses the registry's build_css()
):
    "Render markup in an isolated, replayable notebook preview"
    if height is None: fts += (_sizer,)
    style = f'width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:{height or "auto"};'
    frame = Iframe(srcdoc=demo_html(*fts, css=css), title='answerui preview', style=style)
    replay = Button('↻', type='button', aria_label='replay', title='replay',
        onclick=r"const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc",
        style='border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px')
    return Div(Div(replay, style='text-align:right'), frame)

A preview does not require a registered component; Here `css=''` replaces the registry's stylesheet with an empty one, showing a native element exactly as the browser draws it:

In [13]:
demo(Details(Summary('More information'), P('This disclosure is a native details element.')), css='')

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [{sel: "input[type=range]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { el.style.setProperty(&#39;--val&#39;, el.value) }},
{sel: "input[type=range]", ev: "input", cap: false, deep: false, async fn(el, e) { el.style.setProperty(&#39;--val&#39;, el.value) }},
{sel: "form", ev: "reset", cap: false, deep: false, async fn(el, e) { 
await new Promise(r =&gt; setTimeout(r, 0));
for (const input of el.elements)
  if (input.matches(&#39;input[type=range]&#39;)) input.style.setProperty(&#39;--val&#39;, input.value);
 }}];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
&lt;details&gt;&lt;summary&gt;More information&lt;/summary&gt;      &lt;p&gt;This disclosure is a native details element.&lt;/p&gt;
&lt;/details&gt;&lt;script&gt;new ResizeObserver(() =&gt; frameElement.style.height = document.body.scrollHeight+24+&#39;px&#39;).observe(document.body)&lt;/script&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:auto;"></iframe></div>

Let's recreate the notice from the earlier notebooks. Its styles and behaviors register together. The dismiss button uses an ordinary click rule; `autodismiss` starts a timer when htmx loads the notice.

In [15]:
notice_css = r"""
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
"""
NoticeBox = hypelt('NoticeBox', css=notice_css, layer='components', on={
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()",
    ('&[autodismiss]', ':load'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss') || 4000)); el.remove()"},
    doc='A dismissible notice.')

Click Dismiss, then the ↻ icon to restore the notice:

In [17]:
demo(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*co

This notice disappears after four seconds:

In [19]:
demo(NoticeBox('This notice disappears after four seconds.', autodismiss=4000), height='4rem')

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*co

A custom stylesheet lets us work on a candidate without changing the registry. Here we reuse the notice's CSS but replace its grey background with a pale yellow. The registered behaviors still apply.

In [21]:
candidate_css = notice_css.replace('#f4f4f6', '#fff4cc')
demo(NoticeBox('Trying a warmer surface. ', Button('Dismiss', type='button', dismiss=True)), css=candidate_css)

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #fff4cc;
  border: 1px solid #ccc;
  border-radius: 8px;
}
&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [{sel: "input[type=range]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { el.style.setProperty(&#39;--val&#39;, el.value) }},
{sel: "input[type=range]", ev: "input", cap: false, deep: false, async fn(el, e) { el.style.setProperty(&#39;--val&#39;, el.value) }},
{sel: "form", ev: "reset", cap: false, deep: false, async fn(el, e) { 
await new Promise(r =&gt; setTimeout(r, 0));
for (const input of el.elements)
  if (input.matches(&#39;input[type=range]&#39;)) input.style.setProperty(&#39;--val&#39;, input.value);
 }},
{sel: "notice-box [dismiss]", ev: "click", cap: false, deep: false, async fn(el, e) { el.closest(&#39;notice-box&#39;).remove() }},
{sel: "notice-box[autodismiss]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { await new Promise(r =&gt; setTimeout(r, +el.getAttribute(&#39;autodismiss&#39;) || 4000)); el.remove() }}];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
&lt;notice-box&gt;Trying a warmer surface. &lt;button type="button" dismiss&gt;Dismiss&lt;/button&gt;&lt;/notice-box&gt;&lt;script&gt;new ResizeObserver(() =&gt; frameElement.style.height = document.body.scrollHeight+24+&#39;px&#39;).observe(document.body)&lt;/script&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:auto;"></iframe></div>

An application override takes a different path. Registering CSS in the `app` layer makes it part of subsequent previews and application pages. This rule leaves the notice's definition and behavior alone:

In [23]:
Rules('demo-notice-corners', 'notice-box { border-radius: 1rem; }', doc='Rounder notice corners.')
demo(NoticeBox('The current stylesheet includes the application override.'))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*co

## Reusable examples

A component can provide an example for someone exploring the library. We attach a no-argument markup function to its constructor with `example`. Calling `NoticeBox.demo()` calls that function and passes the fresh markup to `demo`.

Attaching another example replaces the previous one; several variants can be composed in one function.

In [25]:
def example(fn):
    "Decorate a no-argument markup function to provide fn.demo()"
    def _attach(f):
        fn.demo = lambda **kwargs: demo(f(), **kwargs)
        return f
    return _attach


In [27]:
@example(NoticeBox)
def notice_example():
    return Div(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)),
        NoticeBox('This notice stays visible.'), style='display:grid;gap:1rem')

In [29]:
NoticeBox.demo()


<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9646 clamp(0, 0.01797 + 0.00133*cos((1*var(--hue) - 133.91)*1deg) + 0.00071*cos((2*var(--hue) - 305.0)*1deg), 0.02) var(--hue)), oklch(0.1736 0.02 var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9243 0.02 var(--hue)), oklch(0.2291 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7684 0.02 var(--hue)), oklch(0.377 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6318 0.02 var(--hue)), oklch(0.4965 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5083 0.02 var(--hue)), oklch(0.6185 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4053 0.02 var(--hue)), oklch(0.734 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.894 0.04667 var(--hue)), oklch(0.2611 clamp(0, 0.08118 + 0.03727*cos((1*var(--hue) - 324.0)*1deg) + 0.01857*cos((2*var(--hue) - 261.64)*1deg) + 0.01236*cos((3*var(--hue) - 100.29)*1deg) + 0.00447*cos((4*var(--hue) - 81.53)*1deg) + 0.00295*cos((5*var(--hue) - 330.35)*1deg) + 0.00454*cos((6*var(--hue) - 161.37)*1deg) + 0.00108*cos((7*var(--hue) - 148.64)*1deg) + 0.00158*cos((8*var(--hue) - 3.5)*1deg) + 0.00264*cos((9*var(--hue) - 235.89)*1deg) + 0.00067*cos((10*var(--hue) - 214.57)*1deg) + 0.00118*cos((11*var(--hue) - 75.82)*1deg), 0.13557) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7268 clamp(0, 0.13255 + 0.00897*cos((1*var(--hue) - 20.54)*1deg), 0.14154) var(--hue)), oklch(0.418 clamp(0, 0.12579 + 0.05419*cos((1*var(--hue) - 329.06)*1deg) + 0.02532*cos((2*var(--hue) - 268.1)*1deg) + 0.01691*cos((3*var(--hue) - 86.53)*1deg) + 0.0038*cos((4*var(--hue) - 84.49)*1deg) + 0.00355*cos((5*var(--hue) - 304.18)*1deg) + 0.00769*cos((6*var(--hue) - 156.06)*1deg) + 0.00162*cos((7*var(--hue) - 155.96)*1deg) + 0.00185*cos((8*var(--hue) - 5.46)*1deg) + 0.00344*cos((9*var(--hue) - 226.4)*1deg), 0.19199) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.5896 clamp(0, 0.1523 + 0.03949*cos((1*var(--hue) - 339.6)*1deg) + 0.01699*cos((2*var(--hue) - 266.54)*1deg) + 0.02168*cos((3*var(--hue) - 73.38)*1deg) + 0.00344*cos((4*var(--hue) - 190.05)*1deg) + 0.00323*cos((5*var(--hue) - 280.54)*1deg) + 0.00629*cos((6*var(--hue) - 140.44)*1deg) + 0.00341*cos((7*var(--hue) - 287.74)*1deg), 0.19043) var(--hue)), oklch(0.5444 clamp(0, 0.14946 + 0.04722*cos((1*var(--hue) - 335.04)*1deg) + 0.01895*cos((2*var(--hue) - 273.79)*1deg) + 0.02214*cos((3*var(--hue) - 73.51)*1deg) + 0.00176*co

The constructor still emits ordinary markup. Its `.demo()` callable accepts the preview options, such as `height` and `css`. Each call builds a new document with the current styles and behaviors. Redefining the constructor with `hypelt` gives a new function without `.demo`, so rerun the example cell after it.

For a shipped component, the decorated example function should be exported in its component notebook. Importing that module then attaches `.demo()` without executing any example or rendering a preview.

## Export

In [31]:
#| hide
import nbdev; nbdev.nbdev_export()